In [3]:
import pulp as plp
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from tqdm import tqdm


In [4]:
L, W = 30, 40

In [5]:
caixas = [(5, 4), (10, 6), (7, 7)]
caixas

[(5, 4), (10, 6), (7, 7)]

In [6]:
qtds_minimas = [4, 4, 1]

In [7]:
ls = []
ws = []
for caixa in caixas:
    ls.append(caixa[0])
    ls.append(caixa[1])
    ws.append(caixa[1])
    ws.append(caixa[0])
arr_ls = np.array(ls)
arr_ws = np.array(ws)
print("arr_ls:", arr_ls)
print("arr_ws:", arr_ws)

arr_ls: [ 5  4 10  6  7  7]
arr_ws: [ 4  5  6 10  7  7]


In [ ]:
def plotar_caixa(bottom_left: list,
                 l: int,
                 w: int,
                 color: str,
                 label: str):
    x0, y0 = bottom_left
    plt.scatter(x=x0, y=y0, color=color)
    plt.plot([x0, x0 + l], [y0, y0], color=color)
    plt.plot([x0, x0], [y0, y0 + w], color=color)
    plt.plot([x0, x0 + l], [y0 + w, y0 + w], color=color)
    plt.plot([x0 + l, x0 + l], [y0, y0 + w], color=color)
    plt.text(x=x0 + l/2, y=y0 + w/2, s=label, color=color,
             fontsize=9, ha="center", va="center")

def plotar_palete(L: int, W: int, step: int = 1,
                  figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.plot([0, L], [0, 0], "k-")
    plt.plot([0, 0], [0, W], "k-")
    plt.plot([0, L], [W, W], "k-")
    plt.plot([L, L], [0, W], "k-")
    plt.xticks(np.arange(0, L + 1, step))
    plt.yticks(np.arange(0, W + 1, step))
    plt.grid()

plotar_palete(L, W, step=1)
plt.axis("equal")
plotar_caixa(bottom_left=[2, 1], l=caixas[0][0], w=caixas[0][1], color="blue", label="1")
plotar_caixa(bottom_left=[10,4], l=caixas[1][1], w=caixas[1][0], color="red", label="2")

In [ ]:
f_um = lambda arr_n, arr_l: arr_n @ arr_l
f_dois = lambda arr_n, arr_w: arr_n @ arr_w

In [ ]:
menor_lado = min(caixas, key=lambda x: x[1])[1]
menor_lado

In [ ]:
H = []
for ns in tqdm(product(*[list(range(L)) for _ in range(2 * len(caixas))]), total=(L ** (2 * len(caixas)))):
    arr_ns = np.array(ns)
    p = f_um(arr_n=arr_ns, arr_l=arr_ls)
    if p <= L - menor_lado:
        if p not in H:
            H.append(int(p))

H = sorted(H)
H


In [ ]:
V = []
for ns in tqdm(product(*[list(range(L)) for _ in range(2 * len(caixas))]), total=(L ** (2 * len(caixas)))):

    arr_ns = np.array(ns)

    q = f_dois(arr_n=arr_ns, arr_w=arr_ws)
    # print(n_um, n_dois, q)
    if q <= W - menor_lado:

        if q not in V:
            V.append(int(q))

V = sorted(V)
V

In [ ]:
lista_rs = list(product(H, V))
display(lista_rs)

In [ ]:
lista_variaveis = []
for (p, q), i, id_caixa in product(lista_rs, [1, 2], range(len(caixas))):
    ls = [caixas[id_caixa][0], caixas[id_caixa][1]]
    ws = [caixas[id_caixa][1], caixas[id_caixa][0]]

    if p + ls[i-1] <= L and q + ws[i-1] <= W: # checa se cabe no palete
        lista_variaveis.append((id_caixa, i, p, q))
display(lista_variaveis)

In [ ]:
prob = plp.LpProblem(name="Palete-Caixa",
                     sense=plp.LpMaximize)
display(prob)

In [ ]:
dict_variaveis = plp.LpVariable.dicts(name="C", indices=lista_variaveis,
                                      lowBound=0, upBound=1,
                                      cat=plp.LpBinary)
display(dict_variaveis)

In [ ]:
# dict_variaveis.values()
prob += sum(dict_variaveis.values())
display(prob)

In [ ]:
print(f"Qtd variaveis: {prob.numVariables()}")
print(f"Qtd restricoes: {prob.numConstraints()}")

In [ ]:
def ponto_dentro_caixa(ponto_rs: tuple,
                       ponto_pq: tuple,
                       l: int, w: int) -> bool:
    r, s = ponto_rs
    p, q = ponto_pq
    return p <= r < p + l and q <= s < q + w
print(ponto_dentro_caixa((6, 2),
                         (6, 0),
                         l=ls[0], w=ws[0]))

In [ ]:
n = 0
for r, s in lista_rs:
    caixas_sobrepostas = []
    for id_caixa, i, p, q in lista_variaveis:
        ls = [caixas[id_caixa][0], caixas[id_caixa][1]]
        ws = [caixas[id_caixa][1], caixas[id_caixa][0]]

        if ponto_dentro_caixa((r, s),
                              (p, q),
                              l=ls[i-1], w=ws[i-1]):

            caixas_sobrepostas.append(dict_variaveis[(id_caixa, i, p, q)])

    prob += sum(caixas_sobrepostas) <= 1, f"Restricao_1.{n}"
    n += 1
display(prob)

In [ ]:
print(prob.numVariables())
print(prob.numConstraints())

In [ ]:
for idx, qtd_minima in enumerate(qtds_minimas):
    prob += sum([dict_variaveis[(id_caixa, i, p, q)] 
                 for id_caixa, i, p, q in lista_variaveis 
                 if id_caixa == idx]) >= qtd_minima, f"Restricao_2.{idx}"
prob

In [ ]:
print(prob.numVariables())
print(prob.numConstraints())

In [ ]:
plp.listSolvers(onlyAvailable=True)

In [ ]:
solver = plp.getSolver("PULP_CBC_CMD", timeLimit=None)
prob.solve(solver)
print(f"Status: {plp.LpStatus[prob.status]}")

In [ ]:
print(prob.objective.value())

In [ ]:
dict_traducao = \
    dict(zip(dict_variaveis.values(), 
            dict_variaveis.keys()))
display(dict_traducao)

In [ ]:
variaveis_organizadas = []
for variavel in prob.variables():
    # print(variavel.name, variavel.varValue)
    if variavel.varValue > 0:
        var_traduzida = dict_traducao[variavel]
        variaveis_organizadas.append(var_traduzida)
display(variaveis_organizadas)

In [ ]:
variaveis_organizadas = sorted(variaveis_organizadas,
                               key=lambda x: (x[2], x[3]))

In [ ]:
if plp.LpStatus[prob.status] == "Optimal":
    fig, ax = plt.subplots(figsize=(10, 10))
    fig.set_facecolor('#f4f4f4')

    # a = plt.gca()
    # a.set_xticks(list(range(0, L + 1, l_um)) + [L])
    # a.set_yticks(list(range(0, W + 1, w_um)) + [W])
    # a.set_xticklabels(a.get_xticks())
    # a.set_yticklabels(a.get_yticks())

    qtd_caixas = len(variaveis_organizadas)
    qtd_caixas_tipo = [len([v for v in variaveis_organizadas if v[0] == i]) for i in range(len(caixas))]

    from matplotlib import colors as mcolors

    cmap = plt.get_cmap("tab20", len(caixas)) if len(caixas) <= 20 else plt.get_cmap("hsv", len(caixas))
    paleta_caixas = {}
    for id_caixa in range(len(caixas)):
        paleta_caixas[id_caixa] = mcolors.to_hex(cmap(id_caixa))

    for indice_caixa, variavel_organizada in enumerate(variaveis_organizadas, 1):
        id_caixa, i, p, q = variavel_organizada

        ls = [caixas[id_caixa][0], caixas[id_caixa][1]]
        ws = [caixas[id_caixa][1], caixas[id_caixa][0]]

        cor = paleta_caixas.get(id_caixa, "#7f7f7f")
        alpha = 1.0 if i == 1 else 0.9

        ax.add_patch(plt.Rectangle(xy=(p, q), 
                                width=ls[i - 1], 
                                height=ws[i - 1], 
                                facecolor=cor, 
                                edgecolor="white", lw=1,
                                alpha=alpha,))

        plt.text(x=p + ls[i - 1] / 2, 
                y=q + ws[i - 1] / 2, 
                s=f"{indice_caixa}", 
                fontsize=20,
                color="#f4f4f4",
                ha="center", 
                va="center")

    texto_qtds_tipo = "\n".join([f"Quantidade de caixas do tipo {i}: {qtd_caixas_tipo[i]} {caixas[i]}" for i in range(len(caixas))])
    ax.set_title(f"Solução Palete-Caixa\nQuantidade de caixas: {qtd_caixas}\n{texto_qtds_tipo}",
                    fontsize=28)


    legend_handles = []
    for id_caixa in range(len(caixas)):
        patch = plt.Rectangle((0, 0), 1, 1, facecolor=paleta_caixas[id_caixa], edgecolor="white", label=f"Tipo {id_caixa} {caixas[id_caixa]}")
        legend_handles.append(patch)

    ax.legend(handles=legend_handles, loc="upper right", frameon=True, fontsize=12)

    ax.set_xlim(0, L)
    ax.set_ylim(0, W)
    ax.set_aspect("equal")

    ax.xaxis.set_tick_params(labelsize=18)
    ax.yaxis.set_tick_params(labelsize=18)
else:
    print(plp.LpStatus[prob.status])